# FIT4112 · Lab 2 · Notebook 02 — So sánh Apriori và FP-Growth

**Nguyên tắc so sánh công bằng:** cùng basket, cùng `MIN_SUPPORT`, `MAX_LEN`, `MIN_CONFIDENCE`. Hai thuật toán chính xác phải tạo cùng tập mục và luật với cùng chỉ số (trong sai số số học nhỏ).

**Cảnh báo:** dataset demo quá nhỏ để kết luận về hiệu năng. Runtime đo chỉ mang tính minh họa, không cần FP-Growth nhanh hơn.

Notebook này có thể chạy độc lập, không cần output của notebook 01.

## 1. Đọc cùng dữ liệu và cấu hình cho cả hai thuật toán

In [ ]:
from pathlib import Path
from time import perf_counter
import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

def find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "data" / "basket_demo.csv").exists():
            return p
    raise FileNotFoundError("Không thấy data/basket_demo.csv. Hãy mở notebook từ repo hoặc notebooks/.")

ROOT = find_root()
OUTPUT = ROOT / "data" / "processed"
OUTPUT.mkdir(exist_ok=True, parents=True)

def load_basket(file_name):
    path = ROOT / file_name
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy {path}; hãy kiểm tra đường dẫn.")
    df = pd.read_csv(path, index_col=0)
    if df.empty or df.shape[1] < 2:
        raise ValueError("Basket cần ít nhất 1 hóa đơn và 2 sản phẩm.")
    numeric = df.apply(pd.to_numeric, errors="raise")
    if not numeric.isin([0, 1]).all().all():
        raise ValueError("Basket chỉ được chứa các giá trị 0 hoặc 1.")
    return numeric.astype(bool)

def mine_rules(itemsets, min_confidence):
    if itemsets.empty or not itemsets["itemsets"].map(len).ge(2).any():
        return pd.DataFrame(columns=["antecedents", "consequents", "support", "confidence", "lift"])
    return association_rules(itemsets, metric="confidence", min_threshold=min_confidence)

def show_rules(rules):
    cols = ["antecedents_str", "consequents_str", "support", "confidence", "lift"]
    if rules.empty:
        return pd.DataFrame(columns=cols)
    out = rules.copy()
    out["antecedents_str"] = out["antecedents"].map(lambda x: ", ".join(sorted(x)))
    out["consequents_str"] = out["consequents"].map(lambda x: ", ".join(sorted(x)))
    return out[cols].sort_values(["lift", "confidence"], ascending=[False, False]).reset_index(drop=True)

print("Project root:", ROOT)

BASKET_FILE = "data/basket_demo.csv"
MIN_SUPPORT = 0.10
MIN_CONFIDENCE = 0.30
MAX_LEN = 3
REPEATS = 5  # quan sát thời gian tham khảo, không phải benchmark nghiên cứu
basket = load_basket(BASKET_FILE)
print(f"Basket: {basket.shape[0]} hóa đơn × {basket.shape[1]} sản phẩm")
display(basket.head().astype(int))

## 2. Chạy hai thuật toán – cùng điều kiện

Lưu ý: khác biệt **thứ tự xuất hiện các dòng** không phải khác biệt kết quả. Để đối chiếu, chuẩn hóa mỗi itemset thành một bộ tên sản phẩm đã sắp xếp.

In [ ]:
ap_items = apriori(basket, min_support=MIN_SUPPORT, max_len=MAX_LEN, use_colnames=True)
fp_items = fpgrowth(basket, min_support=MIN_SUPPORT, max_len=MAX_LEN, use_colnames=True)
ap_rules = mine_rules(ap_items, MIN_CONFIDENCE)
fp_rules = mine_rules(fp_items, MIN_CONFIDENCE)

def itemsets_map(frame):
    return {tuple(sorted(items)): float(support)
            for support, items in zip(frame["support"], frame["itemsets"])}

def rules_map(frame):
    return {(tuple(sorted(a)), tuple(sorted(b))): (float(s), float(c), float(l))
            for a, b, s, c, l in zip(frame["antecedents"], frame["consequents"],
                                      frame["support"], frame["confidence"], frame["lift"])}

a, b = itemsets_map(ap_items), itemsets_map(fp_items)
ar, fr = rules_map(ap_rules), rules_map(fp_rules)
itemsets_match = a.keys() == b.keys() and all(abs(a[k]-b[k]) < 1e-9 for k in a)
rules_match = ar.keys() == fr.keys() and all(
    all(abs(x-y) < 1e-9 for x,y in zip(ar[k],fr[k])) for k in ar
)
summary = pd.DataFrame([
    {"Thuật toán": "Apriori", "Frequent itemsets": len(ap_items), "Luật": len(ap_rules)},
    {"Thuật toán": "FP-Growth", "Frequent itemsets": len(fp_items), "Luật": len(fp_rules)},
])
display(summary)
print("Trùng tập mục và support?", itemsets_match)
print("Trùng luật và 3 chỉ số?", rules_match)
assert itemsets_match and rules_match, "Kết quả chưa tương đương – kiểm tra dữ liệu/tham số."

## 3. Thử hai ngưỡng min_support

Ở đây chạy **cả hai thuật toán** và ghi số tập mục, số luật. Thay ngưỡng support thường thay đổi kết quả của cả hai theo cùng chiều.

In [ ]:
results = []
for threshold in [0.10, 0.20]:
    for name, algorithm in [("Apriori", apriori), ("FP-Growth", fpgrowth)]:
        fi = algorithm(basket, min_support=threshold, max_len=MAX_LEN, use_colnames=True)
        rr = mine_rules(fi, MIN_CONFIDENCE)
        results.append({"min_support": threshold, "Thuật toán": name,
                        "Frequent itemsets": len(fi), "Luật": len(rr)})
comparison = pd.DataFrame(results)
display(comparison)
assert comparison.groupby("min_support")["Frequent itemsets"].nunique().eq(1).all()
assert comparison.groupby("min_support")["Luật"].nunique().eq(1).all()
print("Cả hai thuật toán có số lượng itemsets/luật giống nhau ở cả hai ngưỡng.")

## 4. So sánh thời gian (tham khảo)

Mỗi lần chạy là một quan sát; kết quả còn phụ thuộc kích thước dữ liệu, hệ thống, phiên bản thư viện và nhiễu thời gian. **Không dùng kết quả của 40 hóa đơn để kết luận FP-Growth luôn nhanh hơn Apriori.**

In [ ]:
from statistics import median
timing = []
for name, algorithm in [("Apriori", apriori), ("FP-Growth", fpgrowth)]:
    durations = []
    for i in range(REPEATS):
        start = perf_counter()
        _ = algorithm(basket, min_support=MIN_SUPPORT, max_len=MAX_LEN, use_colnames=True)
        durations.append((perf_counter() - start) * 1000)
    timing.append({"Thuật toán": name, "Thời gian trung vị (ms)": median(durations),
                   "Số lượt đo": REPEATS})
times = pd.DataFrame(timing)
display(times.round(3))
ax = times.plot.bar(x="Thuật toán", y="Thời gian trung vị (ms)", legend=False,
                    figsize=(6, 4), title="Thời gian minh họa trên dữ liệu demo")
ax.set_ylabel("Milliseconds (ms)")

## 5. Đọc hai bảng luật để hiểu kết quả giống nhau

Có thể khác thứ tự hiển thị, nên hãy quan sát cùng cặp sản phẩm A → B và so sánh các chỉ số.

In [ ]:
print("Apriori")
display(show_rules(ap_rules).head(7).round(3))
print("FP-Growth")
display(show_rules(fp_rules).head(7).round(3))

## 6. Câu hỏi báo cáo

1. Tại mỗi ngưỡng 0.10 và 0.20, số tập mục và số luật của hai thuật toán có bằng nhau không?
2. Đối chiếu tập mục bằng tên sản phẩm, support; đối chiếu **luật có hướng** A → B và support/confidence/lift.
3. Ở lần chạy này thuật toán nào nhanh hơn? Vì sao không nên kết luận tổng quát từ demo nhỏ?
4. Nếu muốn đánh giá hiệu năng nghiêm túc hơn, cần thay đổi dữ liệu và quy trình đo như thế nào?

**Nộp:** hai notebook có output cùng báo cáo ngắn theo `docs/REPORT_TEMPLATE.md`.